# Preparation

In [ ]:
%pip install pandas pyarrow emoji indoNLP transformers torch scikit-learn imbalanced-learn tqdm

In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
import torch

from indoNLP.preprocessing import replace_slang, replace_word_elongation

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import RandomOverSampler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

# Load Data

In [ ]:
df = pd.read_parquet("ulasan_dana.parquet")

In [ ]:
df = df.dropna(subset=["ulasan", "rating"]).reset_index(drop=True)

# WEAK SUPERVISION (LABELING)

In [ ]:
def rating_to_label(r):
    if r <= 2:
        return "negatif"
    elif r == 3:
        return "netral"
    else:
        return "positif"

df["label"] = df["rating"].apply(rating_to_label)


# EMOJI → TEXT

In [ ]:
def demojize_text(text):
    if pd.isna(text) or text == "":
        return text
    return emoji.demojize(text, language="id")

# CUSTOM SLANG MAP (FINTECH)

In [ ]:
custom_slang_map = {
    "apk": "aplikasi",
    "otp": "kode verifikasi",
    "topup": "isi saldo",
    "gak": "tidak",
    "udah": "sudah",
    "udh": "sudah",
    "kalo": "kalau"
}

# CLEAN TEXT FUNCTION (FINAL)

In [ ]:
def clean_text_advanced(text):
    if pd.isna(text) or text == "":
        return text

    text = text.lower()

    # URL removal
    text = re.sub(r"http\S+|www\S+", " ", text)

    # mention & hashtag
    text = re.sub(r"@\w+|#\w+", " ", text)

    # emoji to Indonesian text
    text = demojize_text(text)

    # word elongation
    text = replace_word_elongation(text)

    # slang normalization (indoNLP)
    text = replace_slang(text)

    # custom slang
    for slang, formal in custom_slang_map.items():
        text = re.sub(rf"\b{slang}\b", formal, text)

    # remove non-alphabetic (keep : for emoji tokens)
    text = re.sub(r"[^a-z\s:]", " ", text)

    # whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text

# APPLY PREPROCESSING

In [ ]:
df["text_clean"] = df["review_text"].apply(clean_text_advanced)

# LABEL ENCODING

In [ ]:
le = LabelEncoder()
df["label_enc"] = le.fit_transform(df["label"])

# HANDLE IMBALANCE (ROS)

In [ ]:
X = df[["text_clean"]]
y = df["label_enc"]

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

df_balanced = pd.DataFrame({
    "text_clean": X_res["text_clean"],
    "label_enc": y_res
})

# TRAIN / VALID SPLIT

In [ ]:
train_df, val_df = train_test_split(
    df_balanced,
    test_size=0.2,
    stratify=df_balanced["label_enc"],
    random_state=42
)

# TOKENIZER INDOBERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "indobenchmark/indobert-base-p1"
)

# TOKENIZATION

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text_clean"],
        truncation=True,
        max_length=128
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df).map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text_clean"])
val_ds   = val_ds.remove_columns(["text_clean"])

train_ds.set_format("torch")
val_ds.set_format("torch")

# MODEL + LAYER FREEZING

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1",
    num_labels=len(le.classes_)
)

# Freeze bottom 6 layers
for param in model.bert.encoder.layer[:6].parameters():
    param.requires_grad = False